In [2]:
import numpy as np

In [4]:
np.random.seed(42)
returns_matrix = np.random.randn(10, 5) * 0.02 # shape: (10,5) / assume daily variability of 2%
means = returns_matrix.mean(axis=0) # shape: (5,)
weights = np.array([0.3,0.2,0.2,0.15,0.15]) # shape: (5,)

In [5]:
# de-meaning
demeaned = returns_matrix - means
print(demeaned.shape) # shape: (10,5)

(10, 5)


#### Tracking Broadcasting Rules

1) Rank matching: The shape of returns_matrix is (10,5) and means is (5,). A dimension of size 1 is prepended to the left of means, transforming its shape from (5,) to (1,5)

2) Trailing dimension comparison: Trailing dimension is the rightmost dimension in shape.The trailing dimension of returns_matrix matches with the means. Moving left, the next dimensions are 10 and 1. According to Numpy rules, a dimension of size 1 is compatible with 10.

3) Virtual expansion: The prepended axis of means is virtually stretched from 1 to 10. No actual data copy occurs in memory, instead generate only view with the stride set to 0.

In [6]:
# weighted rate of return
weighted = returns_matrix * weights
print(weighted.shape) # shape: (10,5)

(10, 5)


In [7]:
# checking if stride equals 0
view = np.broadcast_to(means, (10,5))
print(view.strides)

(0, 8)


#### The meaning of "stride equals 0"

The output of view.strides yields (0, 8). A stride value of 0 on the first axis implies that moving to the next row requires 0 bytes of movement. The value of 8 on the second means 8 bytes of float64. The np.broadcast_to generates a memory "View" rather than a "Copy".

In [14]:
# causing error
try:
	result = returns_matrix - np.array([1,2,3,4,5,6,7,8,9,10]) # shape (10,)
except ValueError as e:
	print(e)

operands could not be broadcast together with shapes (10,5) (10,) 


#### Analysis of ValueError

It raises a ValueError because two dimensions are compatible only if they are equal, or if one of them is 1.
The trailing dimension of returns_matrix is 5, whereas the second array is 10. Since 5 != 10 and at least one of them is not 1, the axis expansion fails.
To debug a ValueError, by utilizing np.newaxis, we can expand the shape of a (10,) vector into a (10,1) column matrix. This forces the trailing dimension to be 1

In [13]:
# Fix example
fixed_result = returns_matrix - np.array([1,2,3,4,5,6,7,8,9,10])[:, np.newaxis] # (10,5) - (10,1)
print(fixed_result.shape) # shape (10,5)

(10, 5)


#### Quant trading value

When the shape of returns_matrix is 252days x 500stocks, differences between implmenting de-meaning in Python for loops and Numpy broadcasting:
- Computational Efficiency: Python for loops execute sequentially at the interpreter level, suffering from dynamic type-checking and memory dynamic management overhead for all 126,000(=252 * 500) iterations.
Numpy ndarray guarantees a homogeneous dtype, so the type check within iteration can be omitted, and broadcasting can be calculated as a single line of code without for loops as long as arrays of different sizes are compatible.
- Memory Footprint: Broadcasting uses the stride=0, requiring zero extra memory allocation for the expansion, resulting in an O(1) space complexity.
- Code Maintainability: Nested loops code is not readable well. Broadcasting reduces the operation to a mathematically intuitive single line of code, maxmizing readability.